# Lambda Functions – Simple Examples

We work through each `*FromLambda` component one by one.
At the end we wire them together into a complete optimisation.

In [ ]:
import numpy as np
import metaheuristic_designer as mhd
import seaborn as sns
import matplotlib.pyplot as plt

rng = mhd.check_rng(42)

## 1. Objective function (Rastrigin)

`ObjectiveFromLambda` takes any function that receives a solution vector
and returns a fitness value.

In [ ]:
def rastrigin(x, A=10.0):
    return A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x))


DIM = 5
objfunc = mhd.ObjectiveFromLambda(
    obj_func=rastrigin,
    dimension=DIM,
    mode="min",
    name="Rastrigin",
    A=10.0,  # extra kwargs are forwarded
)

Check the function with a test value

In [ ]:
objfunc.objective(np.array([1.0, 2.0]))  # quick sanity check

## 2. Initializer (uniform random)

`InitializerFromLambda` receives a function `(rng)` that returns
a single solution vector.

In [ ]:
init = mhd.InitializerFromLambda(
    generator=lambda rng: rng.uniform(-5.12, 5.12, size=DIM),
    dimension=DIM,
    pop_size=100,
    rng=rng,
)

Check generating a random individual

In [ ]:
print(init.generate_random())

## 3. Operator – simple Gaussian mutation

`OperatorFromLambda` has signature
`(population, initializer, rng, **kwargs)`.
It must return a `Population`.  We add zero‑mean noise to every gene.

In [ ]:
def gaussian_mutation(population, rng, scale=0.1):
    noise = rng.normal(0, scale, size=population.genotype_matrix.shape)
    return population.update_genotype(population.genotype_matrix + noise)


mut_op = mhd.OperatorFromLambda(
    operator_fn=gaussian_mutation,
    name="GaussMut",
    rng=rng,
    scale=0.5,
)

Check by mutating a random population

In [ ]:
test_pop = init.generate_population(n_individuals=2)
display(test_pop.genotype_matrix)
mut_test = mut_op.evolve(test_pop)
display(mut_test.genotype_matrix)

## 4. Parent selection – tournament

`ParentSelectionFromLambda` expects `(population, amount, rng, **kwargs)`.
It returns indices of selected individuals.

In [ ]:
def tournament_sel(population, amount, rng, size=3):
    n = len(population.fitness)
    pool = rng.integers(0, n, size=(amount, size))
    best = np.argmax(population.fitness[pool], axis=1)
    return pool[np.arange(amount), best]


parent_sel = mhd.parent_selection.ParentSelectionFromLambda(
    selection_fn=tournament_sel,
    amount=10,  # select half of pop
    rng=rng,
    size=3,
)

## 5. Survivor selection – (mu+lambda)

`SurvivorSelectionFromLambda` receives two `Population` objects and a
random state.  We keep the best `pop_size` individuals from the combined
group.

In [ ]:
def mu_plus_lambda(population, offspring, rng):
    """Select the best `len(population)` individuals from parents + offspring."""
    total_fit = np.concatenate([population.fitness, offspring.fitness])
    n = len(population)
    # Sort descending (higher fitness = better, irrespective of min/max mode)
    best_idx = np.argsort(total_fit)[::-1][:n]
    return best_idx


survivor_sel = mhd.SurvivorSelectionFromLambda(
    selection_fn=mu_plus_lambda,
    rng=rng,
)

## 6. Encoding – identity

`EncodingFromLambda` wraps encode/decode functions.  The identity
encoding passes data through unchanged.

In [ ]:
encoding = mhd.EncodingFromLambda(
    encode_fn=lambda x: x,
    decode_fn=lambda x: x,
)

## 7. Constraint handler – clip and penalty

`ConstraintHandlerFromLambda` can repair solutions and/or compute
penalties.  We clip to [-5.12, 5.12] and apply a quadratic penalty for
violations.  This is attached to the objective function.

In [ ]:
constraint = mhd.ConstraintHandlerFromLambda(
    repair_solution_fn=lambda x: np.clip(x, -5.12, 5.12),
    penalty_fn=lambda x: 10.0 * np.sum(np.maximum(0, np.abs(x) - 5.12) ** 2),
)

# Re‑create the objective function with the constraint
objfunc = mhd.ObjectiveFromLambda(
    obj_func=rastrigin,
    dimension=DIM,
    constraint_handler=constraint,
    mode="min",
    name="Rastrigin (constrained)",
    A=10.0,
)

## 8. Build a search strategy

Because we already wrapped each component in its `*FromLambda` class, we
use the standard `SearchStrategy` constructor to assemble them.

In [ ]:
strategy = mhd.strategies.ShuffledPopulationStrategy(
    name="custom strategy",
    initializer=init,
    operator=mut_op,
    parent_sel=parent_sel,
    survivor_sel=survivor_sel,
    encoding=encoding,
    rng=rng,
)

## 9. Run the algorithm

In [ ]:
algo = mhd.algorithms.Algorithm(objfunc, strategy, max_iterations=100, stop_condition_str="max_iterations", reporter="tqdm", verbose_timer=0)
population = algo.optimize()

## 10. Results

In [ ]:
solution, fitness = population.best_solution()
print(f"Best fitness: {fitness}")
print(f"Best solution: {solution}")

We can also visualize the convergence of the algorithm with the help of seaborn

In [ ]:
# Dataframe with the tracked best objective value across generations.
history_data = algo.history_tracker.to_pandas()

# Plotting functionality.
sns.set_theme(style="whitegrid")
ax = sns.lineplot(history_data, x="iteration", y="best_objective", linewidth=2, color="steelblue")
ax.set_xlabel("Generation", fontsize=12)
ax.set_ylabel("Best objective", fontsize=12)
ax.set_title("Convergence Plot", fontsize=14)
plt.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()